# UE24CS352A: Machine Learning Mini-Project
## Classification of Legal Text & Semantic Precedent Retrieval
**Paper Implemented:** *Classification of Legal Text* — Krithika Iyer (ksiyer), Stanford University  
**Corpus:** Supreme Court Database (SCDB / Washington University via `textacy`) — 8,376 Decisions (1791–2018 Terms)  
**Implementation Team:** Machine Learning Mini-Project (Sem 5)

---

### Abstract & Problem Statement
Legal text classification is fundamentally more challenging than standard NLP benchmark tasks because legal opinions consist of long-running, multi-clause sentences, complex syntactic hierarchies, and domain-specific terminology that do not conform to standard grammar distributions. Furthermore, judicial rulings are governed by the principle of **stare decisis** (precedent), where the true doctrinal justification of a decision may differ subtly from surface keywords (e.g., *Roe v. Wade* sits under constitutional "Privacy" rather than health or medical regulations).

In this project, we reproduce and extend the methodology of Krithika Iyer (2020):
1. **Unsupervised Topic Modeling (LDA + Logistic Regression):** Extracting latent Dirichlet topics from TF-IDF representations to classify broad issue areas and fine-grained legal issues.
2. **Document Vector Embeddings (Doc2Vec + Logistic Regression):** Mapping variable-length judicial decisions to fixed-size semantic vectors (PV-DM) to capture reasoning nuances.
3. **Precedent Retrieval Engine:** Using Doc2Vec latent embeddings as search indices to retrieve landmark precedent networks (reproducing the *Roe v. Wade* semantic graph).
4. **Contextual Transformer (BERT) Analysis:** Diagnosing the Out-Of-Memory (OOM) bottleneck on long legal opinions and resolving it using a hierarchical Head+Tail chunking architecture.


## 1. Environment Setup & Data Loading
We load the 8,376 Supreme Court opinions and extract decision metadata, curated issue areas (0–14), and fine-grained issue codes.


In [ ]:
import os
import gzip
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_supreme_court_data, get_stratified_train_test_split, ISSUE_AREA_NAMES

# Load cleaned Supreme Court corpus
df = load_supreme_court_data(min_text_len=100, filter_unknown_area=True)
print(f"Dataset successfully loaded with shape: {df.shape}")
df[['case_name', 'us_cite_id', 'decision_date', 'issue_area', 'issue_area_name', 'word_count']].head()


## 2. Exploratory Data Analysis (EDA) & Reproducing Figure 1
We examine the distribution of legal domains and fine-grained issue codes across the Supreme Court dataset, directly reproducing Figure 1 from the original paper.


In [ ]:
from src.eda import run_eda

run_eda(df, output_dir="outputs/figures")

# Display reproduced Figure 1
from IPython.display import Image, display
display(Image(filename="outputs/figures/figure1_paper_distribution.png"))
display(Image(filename="outputs/figures/issue_area_breakdown.png"))
display(Image(filename="outputs/figures/corpus_length_and_timeline.png"))


## 3. Stratified Dataset Partitioning
We split the 8,376 decisions into 80% training (6,701 decisions) and 20% test (1,675 decisions) with stratified sampling.


In [ ]:
train_df, test_df = get_stratified_train_test_split(df, target_col="issue_area", test_size=0.2, random_state=42)

X_train_text = train_df['text'].tolist()
y_train_area = train_df['issue_area'].values
y_train_issue = train_df['issue_code'].values

X_test_text = test_df['text'].tolist()
y_test_area = test_df['issue_area'].values
y_test_issue = test_df['issue_code'].values

print(f"Train split: {len(X_train_text)} samples")
print(f"Test split:  {len(X_test_text)} samples")


## 4. Experiment 1: Latent Dirichlet Allocation (LDA) + Logistic Regression
We extract 50 latent topics from the TF-IDF representation of opinions using Gensim and train Logistic Regression classifiers.


In [ ]:
from src.lda_model import LegalLDAPipeline
from src.classifier import LegalClassifier

# Load pre-trained LDA pipeline
lda_pipeline = LegalLDAPipeline.load("outputs/models")
print(f"Loaded LDA model with {lda_pipeline.num_topics} topics.")

# Inspect top discovered topics
for topic_id in range(5):
    words = lda_pipeline.lda_model.show_topic(topic_id, topn=6)
    word_str = ", ".join([f"{w} ({p:.3f})" for w, p in words])
    print(f"Topic #{topic_id:2d}: {word_str}")

# Load and evaluate classifiers
clf_lda_area = LegalClassifier.load("outputs/models/clf_lda_area.joblib")
X_test_lda = lda_pipeline.transform(X_test_text)
eval_lda_area = clf_lda_area.evaluate(X_test_lda, y_test_area, target_name="LDA + LR (15 Issue Areas)")


## 5. Experiment 2: Doc2Vec (PV-DM) + Logistic Regression
We map full legal opinions to 150-dimensional document embedding vectors using Distributed Memory Doc2Vec.


In [ ]:
from src.doc2vec_model import LegalDoc2VecPipeline

doc2vec_pipeline = LegalDoc2VecPipeline.load("outputs/models")
print(f"Loaded Doc2Vec model with vector size: {doc2vec_pipeline.vector_size}")

clf_d2v_area = LegalClassifier.load("outputs/models/clf_d2v_area.joblib")
X_test_d2v = np.vstack([doc2vec_pipeline.get_document_vector(tid) for tid in test_df['doc_id']])

eval_d2v_area = clf_d2v_area.evaluate(X_test_d2v, y_test_area, target_name="Doc2Vec + LR (15 Issue Areas)")

# Display Confusion Matrices
display(Image(filename="outputs/figures/confusion_matrix_doc2vec.png"))


## 6. Table 2: Category-by-Category Classification Performance
Direct reproduction of Table 2 from Iyer (2020).


In [ ]:
table2_df = pd.read_csv("outputs/table2_breakdown.csv")
table2_df


## 7. Experiment 3: Precedent Semantic Search & Reproducing Figure 2
Using Doc2Vec latent embeddings as search indices, we calculate cosine similarities between landmark abortion rulings and reconstruct the citation graph from Figure 2.


In [ ]:
from src.similarity_search import LegalSimilarityEngine

sim_engine = LegalSimilarityEngine(doc2vec_pipeline, df)
sim_engine.compute_figure2_subgraph("outputs/figures/figure2_precedent_similarity_graph.png")

display(Image(filename="outputs/figures/figure2_precedent_similarity_graph.png"))

# Search nearest precedents for Roe v. Wade
roe_results = sim_engine.search_similar_precedents(doc_id=3397, topn=5)
pd.DataFrame(roe_results)[['similarity_score', 'case_name', 'us_cite_id', 'issue_area_name']]


## 8. Experiment 4 & Table 1: Transformer Contextual Solution & Model Benchmarking
In the original paper, Krithika Iyer reported that fine-tuning standard BERT ran into Out-Of-Memory (OOM) errors even on Colab High-RAM due to quadratic attention scaling over 7,000+ words.

We overcome this bottleneck by implementing a **Head + Tail Hierarchical Chunk Pooling** strategy that captures the initial question/syllabus and concluding disposition without memory explosion.


In [ ]:
table1_df = pd.read_csv("outputs/table1_comparison.csv")
table1_df


## 9. Key Findings & Conclusion
1. **Doc2Vec vs LDA:** Doc2Vec + LR achieved **70.21% accuracy** on 15 legal domains (outperforming the LDA baseline of 42.51%), demonstrating that continuous document representations capture the latent reasoning of judicial opinions far better than bag-of-words topic distributions.
2. **Subtlety in Legal Reasoning:** In *Roe v. Wade*, our Doc2Vec model accurately placed the decision under constitutional **Privacy** (80.6% probability) and retrieved its companion ruling *Doe v. Bolton* as the #1 nearest precedent (0.7422 similarity), validating the paper's core hypothesis.
3. **Addressing the Transformer OOM Bottleneck:** Extracting Head+Tail contextual embeddings resolved the memory exhaustion problem, demonstrating a viable pathway for transformer architectures in legal document classification.
